In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import warnings

# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

warnings.simplefilter("ignore")

In [2]:
# Loading the part of the dataset where the sheet information is located
information = pd.read_excel('gov_10a_exp__custom_4037524_spreadsheet.xlsx',
                            sheet_name = None, skiprows = 3, nrows = 4,
                            usecols = 'A, B, C')

# Loading the data
data = pd.read_excel('gov_10a_exp__custom_4037524_spreadsheet.xlsx',
                     sheet_name = None, skiprows = 9)

# Extracting the sheet names from the dataset
keys = list(information.keys())
keys.remove('Summary')
keys.remove('Structure')

information_sheets = {}
dataset = pd.DataFrame()

for key in keys:
    
    df = data[key]
    info_sheet = information[key].T

    # Bringing the sheet information into an appropriate format
    # and storing them into a dictionary
    info_sheet.columns = info_sheet.iloc[0]
    info_sheet.drop(info_sheet.head(2).index,inplace = True)
    info_sheet.reset_index(drop = True, inplace = True)

    information_sheets[key] = info_sheet
    
    # Extracting the category of the information fields
    category_column = info_sheet.columns.values[2]
    category = info_sheet[category_column].iloc[0]

    # Dataset contains some flags, visualized as collumns, which get removed
    df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)

    # Removing the last 5 rows (not needed)
    df.drop(df.tail(5).index,inplace = True)

    # Removing first row since it contains no data
    df.drop(index = 0, inplace = True)

    # Replacing broken header names with their actual name
    df.rename(columns={'TIME': 'Country', 'TIME.1': 'UNIT'}, inplace = True)
    
    
    # Extracting the Year columns
    year_list=list(df.columns)
    year_list.remove('Country')
    year_list.remove('UNIT')

    # Splitting the data based on the measurement unit and also adding the category name
    gdp = df.loc[df['UNIT'] == 'Percentage of gross domestic product (GDP)']
    gdp = pd.melt(gdp, id_vars = ['Country'], value_vars = year_list, var_name = 'Year', value_name = '% GDP', ignore_index=False)
    gdp = gdp.sort_values(by = ['Country', 'Year']).reset_index(drop = True)
    gdp['Category'] = category

    millions = df.loc[df['UNIT'] == 'Million euro']
    millions = pd.melt(millions, id_vars = ['Country'], value_vars = year_list, var_name = 'Year', value_name = 'Million euro', ignore_index=False)
    millions = millions.sort_values(by = ['Country', 'Year']).reset_index(drop = True)

    # Merging the necessary data
    df = millions
    df['% GDP'] = gdp['% GDP']
    df['Category'] = gdp['Category']
    
    dataset = pd.concat([dataset, df])

dataset.reset_index(drop = True, inplace = True)
dataset.replace(':', np.nan, inplace = True)
dataset.replace('European Union - 27 countries (from 2020)', 'European Union', inplace = True)
dataset.replace('Euro area - 19 countries  (from 2015)', 'Eurozone', inplace = True)
dataset.replace('Germany (until 1990 former territory of the FRG)', 'Germany', inplace = True)


####################################################
# Handling the GDP dataset

# Loading the data
data = pd.read_excel('nama_10_gdp__custom_4142114_spreadsheet.xlsx',
                    sheet_name = None, skiprows = 8)

# Extracting the sheet names from the dataset
keys = list(data.keys())
keys.remove('Summary')
keys.remove('Structure')

for key in keys:
    df = data[key]

    # Dataset contains some flags, visualized as collumns, which get removed
    df.drop(df.columns[df.columns.str.contains('unnamed',case = False)],axis = 1, inplace = True)

    # Removing the last 5 rows (not needed)
    df.drop(df.tail(5).index,inplace = True)

    # Removing first row since it contains no data
    df.drop(index = 0, inplace = True)

    # Replacing broken header names with their actual name
    df.rename(columns={'TIME': 'Country', 'TIME.1': 'UNIT'}, inplace = True)

    # Extracting the Year columns
    year_list=list(df.columns)
    year_list.remove('Country')

    gdp = pd.melt(df, id_vars = ['Country'], value_vars = year_list, var_name = 'Year', value_name = 'Million GDP', ignore_index=False)

    gdp.reset_index(drop = True, inplace = True)
    gdp.replace(':', np.nan, inplace = True)
    gdp.replace('European Union - 27 countries (from 2020)', 'European Union', inplace = True)
    gdp.replace('Euro area - 19 countries  (from 2015)', 'Eurozone', inplace = True)
    gdp.replace('Germany (until 1990 former territory of the FRG)', 'Germany', inplace = True)

dataset = dataset.merge(gdp, how='left')

# Updating the %GDP with the new values
dataset['% GDP'] = (dataset['Million euro'] / dataset['Million GDP']) * 100

dataset

,Country,Year,Million euro,% GDP,Category,Million GDP
0,Austria,2012,163191.9,51.213044,Total,318653.0
1,Austria,2013,167292.1,51.647679,Total,323910.2
2,Austria,2014,174671.6,52.430930,Total,333146.1
3,Austria,2015,176030.0,51.131498,Total,344269.2
4,Austria,2016,179059.0,50.071307,Total,357608.0
...,...,...,...,...,...,...
25595,Switzerland,2017,55.1,0.008948,Social protection n.e.c.,615776.3
25596,Switzerland,2018,53.7,0.008742,Social protection n.e.c.,614304.4
25597,Switzerland,2019,56.1,0.008705,Social protection n.e.c.,644443.2
25598,Switzerland,2020,59.3,0.009138,Social protection n.e.c.,648913.3


### Task 1a

In [7]:
# Drop category 'Total' as well as aggregates 'European Union' and 'Eurozone'
dataset = dataset.loc[(dataset['Category'] != 'Total') &
                      (dataset['Country'] != 'European Union') &
                      (dataset['Country'] != 'Eurozone')]


# Countries spending the most (in terms of GDP) per year
highest_gdp = dataset.loc[dataset.groupby(['Year'])['% GDP'].idxmax()]

highest_gdp


,Country,Year,Million euro,% GDP,Category,Million GDP
22460,Denmark,2012,62576.4,24.580443,Social protection,254578.0
22501,Finland,2013,50262.0,24.599527,Social protection,204321.0
22502,Finland,2014,52184.0,25.222212,Social protection,206897.0
22503,Finland,2015,53413.0,25.268113,Social protection,211385.0
22504,Finland,2016,55426.0,25.481110,Social protection,217518.0
22505,Finland,2017,55546.0,24.545185,Social protection,226301.0
22506,Finland,2018,56618.0,24.251484,Social protection,233462.0
22507,Finland,2019,57720.0,24.064238,Social protection,239858.0
22518,France,2020,627916.0,27.176993,Social protection,2310469.0
22409,Austria,2021,88969.6,21.905671,Social protection,406148.7


### Task 1b

In [8]:
# Countries spending the most (absolute values in mln EURO) per year
highest_absolute = dataset.loc[dataset.groupby(['Year'])['Million euro'].idxmax()]

highest_absolute

,Country,Year,Million euro,% GDP,Category,Million GDP
22520,Germany,2012,518918.0,18.901982,Social protection,2745310.0
22521,Germany,2013,533910.0,18.991232,Social protection,2811350.0
22522,Germany,2014,550151.0,18.792969,Social protection,2927430.0
22523,Germany,2015,577822.0,19.094105,Social protection,3026180.0
22524,Germany,2016,611468.0,19.506179,Social protection,3134740.0
22525,Germany,2017,634997.0,19.435748,Social protection,3267160.0
22526,Germany,2018,648812.0,19.278611,Social protection,3365450.0
22527,Germany,2019,681481.0,19.620789,Social protection,3473260.0
22528,Germany,2020,734813.0,21.577686,Social protection,3405430.0
22409,Austria,2021,88969.6,21.905671,Social protection,406148.7
